In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
!pip install emoji

In [ ]:
import html
import emoji

In [ ]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import tensorflow as tf
import keras
import keras_nlp
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix, f1_score
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from keras.callbacks import EarlyStopping, ReduceLROnPlateau

# Input data files are available in the read-only "../input/" directory
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# Make sure the competition data files are found
competition_path = '/kaggle/input/nlp-getting-started'
train_path = os.path.join(competition_path, 'train.csv')
test_path = os.path.join(competition_path, 'test.csv')
submission_path = os.path.join(competition_path, 'sample_submission.csv')

print(f"\nChecking if competition files exist:")
print(f"Train file exists: {os.path.exists(train_path)}")
print(f"Test file exists: {os.path.exists(test_path)}")
print(f"Sample submission exists: {os.path.exists(submission_path)}")


import re
import string
import os

print("TensorFlow version:", tf.__version__)
print("KerasNLP version:", keras_nlp.__version__)

# ===== 独自のストップワードリスト（NLTK依存を避ける）=====
STOPWORDS = set([
    'i', 'me', 'my', 'myself', 'we', 'our', 'ours', 'ourselves', 'you', 
    'your', 'yours', 'yourself', 'yourselves', 'he', 'him', 'his', 'himself', 
    'she', 'her', 'hers', 'herself', 'it', 'its', 'itself', 'they', 'them', 
    'their', 'theirs', 'themselves', 'what', 'which', 'who', 'whom', 'this', 
    'that', 'these', 'those', 'am', 'is', 'are', 'was', 'were', 'be', 'been', 
    'being', 'have', 'has', 'had', 'having', 'do', 'does', 'did', 'doing', 
    'a', 'an', 'the', 'and', 'but', 'if', 'or', 'because', 'as', 'until', 
    'while', 'of', 'at', 'by', 'for', 'with', 'about', 'against', 'between', 
    'into', 'through', 'during', 'before', 'after', 'above', 'below', 'to', 
    'from', 'up', 'down', 'in', 'out', 'on', 'off', 'over', 'under', 'again', 
    'further', 'then', 'once', 'here', 'there', 'when', 'where', 'why', 'how', 
    'all', 'any', 'both', 'each', 'few', 'more', 'most', 'other', 'some', 'such', 
    'no', 'nor', 'not', 'only', 'own', 'same', 'so', 'than', 'too', 'very', 
    's', 't', 'can', 'will', 'just', 'don', 'should', 'now'
])
CONTRACTION_MAP = {
    "ain't": "is not", "aren't": "are not", "can't": "cannot", "can't've": "cannot have",
    "'cause": "because", "could've": "could have", "couldn't": "could not", "couldn't've": "could not have",
    "didn't": "did not", "doesn't": "does not", "don't": "do not", "hadn't": "had not",
    "hadn't've": "had not have", "hasn't": "has not", "haven't": "have not", "he'd": "he would",
    "he'd've": "he would have", "he'll": "he will", "he'll've": "he will have", "he's": "he is",
    "how'd": "how did", "how'd'y": "how do you", "how'll": "how will", "how's": "how is",
    "I'd": "I would", "I'd've": "I would have", "I'll": "I will", "I'll've": "I will have",
    "I'm": "I am", "I've": "I have", "isn't": "is not", "it'd": "it would",
    "it'd've": "it would have", "it'll": "it will", "it'll've": "it will have", "it's": "it is",
    "let's": "let us", "ma'am": "madam", "mayn't": "may not", "might've": "might have",
    "mightn't": "might not", "mightn't've": "might not have", "must've": "must have",
    "mustn't": "must not", "mustn't've": "must not have", "needn't": "need not",
    "needn't've": "need not have", "o'clock": "of the clock", "oughtn't": "ought not",
    "oughtn't've": "ought not have", "shan't": "shall not", "sha'n't": "shall not",
    "shan't've": "shall not have", "she'd": "she would", "she'd've": "she would have",
    "she'll": "she will", "she'll've": "she will have", "she's": "she is",
    "should've": "should have", "shouldn't": "should not", "shouldn't've": "should not have",
    "so've": "so have", "so's": "so is", "that'd": "that would", "that'd've": "that would have",
    "that's": "that is", "there'd": "there would", "there'd've": "there would have",
    "there's": "there is", "they'd": "they would", "they'd've": "they would have",
    "they'll": "they will", "they'll've": "they will have", "they're": "they are",
    "they've": "they have", "to've": "to have", "wasn't": "was not", "we'd": "we would",
    "we'd've": "we would have", "we'll": "we will", "we'll've": "we will have",
    "we're": "we are", "we've": "we have", "weren't": "were not", "what'll": "what will",
    "what'll've": "what will have", "what're": "what are", "what's": "what is",
    "what've": "what have", "when's": "when is", "when've": "when have",
    "where'd": "where did", "where's": "where is", "where've": "where have",
    "who'll": "who will", "who'll've": "who will have", "who's": "who is",
    "who've": "who have", "why's": "why is", "why've": "why have",
    "will've": "will have", "won't": "will not", "won't've": "will not have",
    "would've": "would have", "wouldn't": "would not", "wouldn't've": "would not have",
    "y'all": "you all", "y'all'd": "you all would", "y'all'd've": "you all would have",
    "y'all're": "you all are", "y'all've": "you all have", "you'd": "you would",
    "you'd've": "you would have", "you'll": "you will", "you'll've": "you will have",
    "you're": "you are", "you've": "you have"
}

def expand_contractions(text, contraction_map=CONTRACTION_MAP):
    # 正規表現で，辞書に定義されたすべての縮約系を検出
    contractions_pattern = re.compile('({})'.format('|'.join(contraction_map.keys())), 
                                      flags=re.IGNORECASE|re.DOTALL)
    
    def expand_match(contraction):
        # マッチした完全な単語を取得
        match = contraction.group(0)
        
        # 対応する完全な表現を辞書から取得
        expanded_contraction = contraction_map.get(match) \
                                if contraction_map.get(match) \
                                else contraction_map.get(match.lower())
        
        # 全文が見つかった場合 (expanded_contractionがNoneでないとき)
        if expanded_contraction:
        
            first_char = match[0]
            if first_char.isupper():
                return first_char + expanded_contraction[1:]
            else:
                return expanded_contraction
        # 辞書に見つからない場合，展開すべき縮約形でない可能性がある（例：”user's)
        else:
        
            return match
        
    expanded_text = contractions_pattern.sub(expand_match, text)
    # 残っている可能性のあるアポストロフィを削除（必要に応じて保持も可能）
    # expanded_text = re.sub("'", "", expanded_text) 
    return expanded_text

# ===== テキストクレンジング関数（BERT用に最適化）=====
def clean_text_for_bert(text):
    
    if not isinstance(text, str):
        return ''
    
    # 1. HTMLエンティティをデコード (例. &amp; → &)
    text = html.unescape(text)
    
    # 2. 絵文字をテキストに変換 (例. 🔥 → :fire:)
    text = emoji.demojize(text, delimiters=(" ", " "))
    
    # 3. 縮約形を展開 (例. "don't" → "do not")
    text = expand_contractions(text)
    
    # 4. URLリンクを削除
    text = re.sub(r'http\S+|www\S+|https\S+', '', text)
    
    # 5. ユーザーのメンション（@ユーザー名）を削除 
    text = re.sub(r'@\w+', '', text)
    
    # 6. ハッシュタグの「#」だけを削除
    text = re.sub(r'#(\w+)', r'\1', text)
    
    # 7.  英数字と基本的な句読点以外の記号を削除
    text = re.sub(r'[^A-Za-z0-9\s.,!?\'`]', '', text)
    
    # 8. 同じ文字の繰り返しを縮める（"soooo" → "soo"）
   
    text = re.sub(r'(.)\1{2,}', r'\1\1', text)
    
    # 9. 余分な空白（スペース）を削除
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

def clean_text_traditional(text):
    
    if isinstance(text, str):
        # Convert to lowercase
        text = text.lower()
        
        # Remove URLs
        text = re.sub(r'http\S+|www\S+|https\S+', '', text)
        
        # Remove user mentions and hashtags
        text = re.sub(r'@\w+|#\w+', '', text)
        
        # Remove punctuation
        text = text.translate(str.maketrans('', '', string.punctuation))
        
        # Remove extra whitespaces
        text = re.sub(r'\s+', ' ', text).strip()
        
        return text
    else:
        return ''

# ===== データの読み込みと基本情報の確認 =====
df_train = pd.read_csv("/kaggle/input/nlp-getting-started/train.csv")
df_test = pd.read_csv("/kaggle/input/nlp-getting-started/test.csv")

print('Training Set Shape = {}'.format(df_train.shape))
print('Training Set Memory Usage = {:.2f} MB'.format(df_train.memory_usage().sum() / 1024**2))
print('Test Set Shape = {}'.format(df_test.shape))
print('Test Set Memory Usage = {:.2f} MB'.format(df_test.memory_usage().sum() / 1024**2))

# ===== 基本的なEDAと欠損値処理 =====
print("\n=== データの探索 ===")
print("Training set columns:", df_train.columns.tolist())
print("\nMissing values in training set:")
print(df_train.isnull().sum())
print("\nTarget distribution:")
print(df_train['target'].value_counts())
print(df_train['target'].value_counts(normalize=True).round(3))

# データの探索
df_train['location'].fillna('unknown', inplace=True)
df_train['keyword'].fillna('unknown', inplace=True)
df_test['location'].fillna('unknown', inplace=True)
df_test['keyword'].fillna('unknown', inplace=True)

# ===== テキストのクリーニング処理を適用 =====
print("\n=== テキストのクリーニング処理を適用 ===")
print("元のテキスト例:")
print(df_train['text'].iloc[0])

#  BERTに最適化したテキストクリーニング関数を適用
df_train['clean_text'] = df_train['text'].apply(clean_text_for_bert)
df_test['clean_text'] = df_test['text'].apply(clean_text_for_bert)

print("\nクリーニング後のテキスト例:")
print(df_train['clean_text'].iloc[0])

# ===== テキストの長さを分析 =====
df_train["length"] = df_train["clean_text"].apply(lambda x : len(x))
df_test["length"] = df_test["clean_text"].apply(lambda x : len(x))

print("\n=== テキスト長の統計情報 ===")
print("Train Length Stat (cleaned text)")
print(df_train["length"].describe())
print()
print("Test Length Stat (cleaned text)")
print(df_test["length"].describe())

# ===== 単語数の統計と可視化 =====
df_train['word_count'] = df_train['clean_text'].apply(lambda x: len(x.split()))

plt.figure(figsize=(10, 6))
sns.histplot(data=df_train, x='word_count', hue='target', bins=30, kde=True)
plt.title('Word Count Distribution by Target (After Cleaning)')
plt.xlabel('Word Count')
plt.ylabel('Frequency')
plt.legend(['Not Disaster', 'Disaster'])
plt.show()

# ===== モデルパラメータ設定 =====
BATCH_SIZE = 32
NUM_TRAINING_EXAMPLES = df_train.shape[0]
TRAIN_SPLIT = 0.8
VAL_SPLIT = 0.2
STEPS_PER_EPOCH = int(NUM_TRAINING_EXAMPLES)*TRAIN_SPLIT // BATCH_SIZE

# =====エポック数（訓練回数）  =====
EPOCHS = 20  
AUTO = tf.data.experimental.AUTOTUNE

# ===== クリーニング後のテキストで学習 =====
X = df_train["clean_text"]  
y = df_train["target"]

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=VAL_SPLIT, random_state=42)

X_test = df_test["clean_text"]  

# Load a DistilBERT model.
preset= "distil_bert_base_en_uncased"

# Use a shorter sequence length.
preprocessor = keras_nlp.models.DistilBertPreprocessor.from_preset(preset,
                                                                   sequence_length=160,
                                                                   name="preprocessor_4_tweets"
                                                                  )

# Pretrained classifier.
classifier = keras_nlp.models.DistilBertClassifier.from_preset(preset,
                                                               preprocessor = preprocessor, 
                                                               num_classes=2)

classifier.summary()

# コンパイル
classifier.compile(
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    optimizer=keras.optimizers.Adam(learning_rate=1e-5),
    metrics= ["accuracy"]  
)

# EarlyStopping（早期終了）のコールバック設定
early_stopping = EarlyStopping(monitor='val_loss', 
                               patience=3, 
                               restore_best_weights=True,
                               verbose=1)

# （学習率減衰のコールバック設定
reduce_lr = ReduceLROnPlateau(monitor='val_loss', 
                              factor=0.5, 
                              patience=2, 
                              min_lr=1e-7, # 最小学習率
                              verbose=1)
# ===== トレーニングの開始 =====
print("\n===　トレーニング開始 ===")
print(f"訓練サンプル数: {len(X_train)}")
print(f"検証サンプル数: {len(X_val)}")
print(f"エポック数: {EPOCHS}")

# トレーニング実行
history = classifier.fit(x=X_train,
                         y=y_train,
                         batch_size=BATCH_SIZE,
                         epochs=EPOCHS, 
                         validation_data=(X_val, y_val),
                         verbose=1,
                         callbacks=[early_stopping, reduce_lr] # 应用回调
                        )

# ===== 学習の推移を可視化 =====
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Model Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Model Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.tight_layout()
plt.show()

# ===== モデルの保存 =====

print("\n=== モデルを保存中... ===")

classifier.save("distilbert_classifier.keras") 
print("モデルは 'distilbert_classifier.keras' に保存されました")
# =====  評価関数 =====
def displayConfusionMatrix(y_true, y_pred, dataset):
    disp = ConfusionMatrixDisplay.from_predictions(
        y_true,
        np.argmax(y_pred, axis=1),
        display_labels=["Not Disaster","Disaster"],
        cmap=plt.cm.Blues
    )

    tn, fp, fn, tp = confusion_matrix(y_true, np.argmax(y_pred, axis=1)).ravel()
    f1_score = tp / (tp+((fn+fp)/2))

    disp.ax_.set_title("Confusion Matrix on " + dataset + " Dataset -- F1 Score: " + str(f1_score.round(2)))
    plt.show()

# ===== モデル評価 =====
print("\n=== モデル評価 ===")
y_pred_train = classifier.predict(X_train)
displayConfusionMatrix(y_train, y_pred_train, "Training")

y_pred_val = classifier.predict(X_val)
displayConfusionMatrix(y_val, y_pred_val, "Validation")

# ===== 提出ファイルの生成 =====
print("\n=== 予測結果の生成 ===")
sample_submission = pd.read_csv("/kaggle/input/nlp-getting-started/sample_submission.csv")
sample_submission.head()

sample_submission["target"] = np.argmax(classifier.predict(X_test), axis=1)
sample_submission.describe()

sample_submission.to_csv("submission.csv", index=False)
print("提出ファイルを保存しました: submission.csv")